In [ ]:
# =====================================================================
# CELL 1: ABSOLUTE TOP OF YOUR NOTEBOOK
# =====================================================================
import os

# FIX 1: Prevent Julia from overriding Python signal handles (Fixes the Segfault)
os.environ["PYTHON_JULIACALL_HANDLE_SIGNALS"] = "yes"

# FIX 2: Restrict Julia threads to keep it completely isolated from JAX/XLA thread pools
os.environ["JULIA_NUM_THREADS"] = "4"

# Force JAX backend configuration
os.environ["CBEAM_BACKEND"] = "jax"

import time
import warnings
import numpy as np
import jax
import jax.numpy as jnp

import cbeam
import matplotlib.pyplot as plt


In [ ]:
# =====================================================================
# CELL 1: IMPORTS AND SYSTEM SETUP
# =====================================================================
# Your refactored custom modules

import time

import jax
import jax.numpy as jnp
from jax import random
from scipy.interpolate import LinearNDInterpolator

from jaxMLP import (
    init_mlp_params, mlp_forward, huber_loss, update_step, get_batches
)

from batch_propagation_pipeline import *

# --- CONSTANTS ---
N_SIGNALS = 19
M_MODES = 5
DATASET_PATH = "lantern_training_data.npz"
#DATASET_PATH = "lantern_training_data_hard.npz"

ideal_grid_positions = [
    (0.0000, 0.0000), (1.0000, 0.0000), (0.5000, 0.8660), (-0.5000, 0.8660),
    (-1.0000, 0.0000), (-0.5000, -0.8660), (0.5000, -0.8660), (2.0000, 0.0000),
    (1.5000, 0.8660), (1.0000, 1.7321), (0.0000, 1.7321), (-1.0000, 1.7321),
    (-1.5000, 0.8660), (-2.0000, 0.0000), (-1.5000, -0.8660), (-1.0000, -1.7321),
    (0.0000, -1.7321), (1.0000, -1.7321), (1.5000, -0.8660)
]

In [ ]:
# =====================================================================
# CELL 2: DATA GENERATION, PROCESSING & I/O
# =====================================================================
LOAD_DATA_FROM_DISK = True  # Set to True to skip simulation and load saved arrays

if LOAD_DATA_FROM_DISK and os.path.exists(DATASET_PATH):
    print(f"Loading dataset from {DATASET_PATH}...")
    data = np.load(DATASET_PATH)
    
    X_train, X_val = data['X_train'], data['X_val']
    Y_train, Y_val = data['Y_train'], data['Y_val']
    X_mean, X_std = data['X_mean'], data['X_std']
    Y_mean, Y_std = data['Y_mean'], data['Y_std']
    
    print(f"Loaded successfully! Train samples: {X_train.shape[0]}, Val samples: {X_val.shape[0]}")

else:
    print("Generating new pipeline dataset...")
    p = get_simulation_parameters()
    
    print("\n" + "="*60)
    print("SIMULATION PARAMETERS")
    print("="*60)
    print(f"Wavelength: {p['wl']} μm")
    print(f"Lantern length: {p['z_ex']} μm")
    print(f"Cladding radius: {p['rclad']} μm")
    print(f"Jacket radius: {p['rjack']} μm")
    print(f"Core radius: {p['rcore']:.3f} μm")
    print(f"Taper factor: {p['taper_factor']}")
    print(f"Available influence function: {p['ifunc_file']}")
    
    # 1. Pipeline Execution
    import specula
    
    print("Initializing SPECULA GPU...")
    specula.init(0)
    from specula.data_objects.ifunc import IFunc
    
    print("Loading influence function...")
    # UPDATE THIS LINE TO EXPLICITLY PASS THE DEVICE INDEX:
    print(p["ifunc_file"])
    ifunc = IFunc.restore(p["ifunc_file"]) 

    prop12 = build_and_characterize_lantern(p)    
    # Pass the correctly restored ifunc object to the pipeline
    pipeline = BatchPropagationPipeline(prop12, p, ifunc)


    N_SAMPLES = 3000    
    # coeff = create_sparse_aberration_configs_mono(n=N_SAMPLES, m=M_MODES, minv=-300, maxv=300)
    coeff = create_random_aberration_configs(n=N_SAMPLES, m=M_MODES, minv=-300, maxv=300)
    
    u0_batch = pipeline.generate_batch_modal_coefficients(coeff, use_gpu=False)
    uf_batch, zs, us_batch = pipeline.propagate_batch(u0_batch)
    E_out_np = np.array(pipeline.reconstruct_batch_output_fields(uf_batch))
    
    # 2. Calibration & Alignment
    mesh_final = prop12.mesh
    intensities_mesh = np.abs(E_out_np)**2 
    mean_unstructured_profile = np.mean(intensities_mesh, axis=0)
    
    core_centers, plot_x_out, plot_y_out, dx, dy = calibrate_subpixel_centers(mean_unstructured_profile, mesh_final)
    ideal_permutation = map_evaluated_to_ideal_geometry(core_centers, ideal_grid_positions)
    aligned_core_centers = np.array(core_centers)[ideal_permutation]
    
    # 3. Interpolation & Extraction
    X_plot_out, Y_plot_out = np.meshgrid(plot_x_out, plot_y_out)
    interp = LinearNDInterpolator(np.array(mesh_final.points)[:, :2], intensities_mesh.T)
    images_jax = jnp.nan_to_num(jnp.array(interp(X_plot_out, Y_plot_out)).transpose(2, 0, 1), nan=0.0)
    
    # Build Coordinates
    x_min, y_min = plot_x_out.min(), plot_y_out.min()
    offsets = np.arange(-2, 3)
    all_coords = []
    for cx, cy in aligned_core_centers:
        sub_cols, sub_rows = np.meshgrid(int(np.round((cx - x_min)/dx)) + offsets, 
                                         int(np.round((cy - y_min)/dy)) + offsets)
        all_coords.append(np.stack([sub_rows, sub_cols], axis=0))
    aligned_precomputed_coords = jnp.array(np.stack(all_coords, axis=1), dtype=jnp.int32)
    
    # Extract
    raw_signals_jax = batch_collect_subpixel_signals(images_jax, aligned_precomputed_coords)
    X_raw, Y_raw = jnp.array(raw_signals_jax), jnp.array(coeff)
    
    # 4. Scaling and Splitting
    X_mean, X_std = jnp.mean(X_raw, axis=0), jnp.std(X_raw, axis=0) + 1e-6
    Y_mean, Y_std = jnp.mean(Y_raw, axis=0), jnp.std(Y_raw, axis=0) + 1e-6
    X_scaled, Y_scaled = (X_raw - X_mean) / X_std, (Y_raw - Y_mean) / Y_std
    
    split_idx = int(0.8 * N_SAMPLES)
    X_train, X_val = X_scaled[:split_idx], X_scaled[split_idx:]
    Y_train, Y_val = Y_scaled[:split_idx], Y_scaled[split_idx:]
    
    # 5. Save Data for Next Time
    print(f"Saving dataset to {DATASET_PATH}...")
    np.savez(DATASET_PATH, X_train=X_train, Y_train=Y_train, X_val=X_val, Y_val=Y_val, 
             X_mean=X_mean, X_std=X_std, Y_mean=Y_mean, Y_std=Y_std)
    print("Dataset generated and saved!")

In [ ]:
# =====================================================================
# CELL 3: MLP TRAINING EXECUTION
# =====================================================================
main_key = random.PRNGKey(42)
init_key, train_key = random.split(main_key, 2)

layer_sizes = [N_SIGNALS, 64, 64, 64, 64, M_MODES]
params = init_mlp_params(layer_sizes, init_key)


In [ ]:
EPOCHS = 20000
BATCH_SIZE = 128
LEARNING_RATE = 1e-2

print(f"\n--- Starting Wavefront Inversion Loop ({EPOCHS} Epochs) ---")
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    start_epoch = time.time()
    train_key, batch_key = random.split(train_key)
    batches = get_batches(X_train, Y_train, BATCH_SIZE, batch_key)
    
    epoch_loss = 0.0
    for X_batch, Y_batch in batches:
        train_key, step_key = random.split(train_key)
        params, loss_val = update_step(params, X_batch, Y_batch, step_key, LEARNING_RATE)
        epoch_loss += loss_val
    epoch_loss /= len(batches)
    
    Y_val_pred = mlp_forward(params, X_val, deterministic=True)
    val_loss = huber_loss(Y_val, Y_val_pred)
    
    train_losses.append(epoch_loss)
    val_losses.append(val_loss)
    
    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {epoch_loss:.5f} | Val Loss: {val_loss:.5f} | Time/Ep: {time.time() - start_epoch:.2f}s")

In [ ]:
# =====================================================================
# CELL 7: DIAGNOSTIC PERFORMANCE ANALYSIS
# =====================================================================
Y_val_pred_scaled = mlp_forward(params, X_val, deterministic=True)
Y_val_pred_physical = (Y_val_pred_scaled * Y_std) + Y_mean
Y_val_true_physical = (Y_val * Y_std) + Y_mean

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Dataset Error')
plt.plot(val_losses, label='Validation Dataset Error')
plt.yscale('log')
plt.xlabel('Training Epochs')
plt.ylabel('Huber Loss Value')
plt.title('Convergence Analysis Profile')
plt.legend()

plt.subplot(1, 2, 2)
target_mode = 2  
plt.scatter(Y_val_true_physical[:, target_mode], Y_val_pred_physical[:, target_mode], alpha=0.4, color='darkmagenta')
plt.plot([Y_val_true_physical.min(), Y_val_true_physical.max()], [Y_val_true_physical.min(), Y_val_true_physical.max()], 'r--', lw=2)
plt.xlabel('Simulated Truth Coefficients (nm)')
plt.ylabel('MLP Reconstructed Output (nm)')
plt.title(f'Wavefront Inversion Precision: Mode #{target_mode}')
plt.tight_layout()
plt.show()

In [ ]:
# !pip install pysr

In [ ]:
# =====================================================================
# PYSR PRE-REQUISITE: INITIALIZE JULIA BACKEND (Run Once)
# =====================================================================
#from juliacall import Main as jl
#
#print("Installing SymbolicRegression backend into your Julia environment...")
#jl.seval('import Pkg; Pkg.add("SymbolicRegression")')
#print("Installation complete! You can now run your PySR cells.")

In [ ]:
# =====================================================================
# ALTERNATIVE CELL 4 & 5: ROCK-SOLID PYSR ENGINE
# =====================================================================
import numpy as np
from pysr import PySRRegressor

# Explicitly clean the JAX matrices into raw float32 NumPy vectors
X_train_np = np.array(X_train, dtype=np.float32)
Y_train_np = np.array(Y_train, dtype=np.float32)

print("\n--- Initializing PySR Multi-Output Symbolic Regressor ---")
pysr_model = PySRRegressor(
    niterations=500,                # High-efficiency iteration depth
    populations=40,                 # Evolution islands
    population_size=40,             
    
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["sin", "cos", "abs", "exp"],
    
    maxsize=20,                     # Keeps equations compact and robust
    nested_constraints={
        "sin": {"sin": 1, "cos": 1}, 
        "cos": {"sin": 1, "cos": 1},
        "exp": {"exp": 0}
    },
    
    model_selection="best",
    random_state=42,
    
    # FIX 3: Use 'serial' or 'multithreading' explicitly without passing 'procs'
    # This prevents worker conflicts entirely.
    parallelism="serial",           
)

print("Evolving explicit equations...")
# Sub-slice the data down to 300 samples to verify backend stability 
# before scaling up to your entire dataset matrix.
pysr_model.fit(X_train_np[:300], Y_train_np[:300])

print("\n=== DISCOVERED PARETO-OPTIMAL EQUATIONS ===")
print(pysr_model.equations_)

In [ ]:
# =====================================================================
# ALTERNATIVE CELL 7: DIAGNOSTIC PERFORMANCE ANALYSIS FOR SR
# =====================================================================
# Predict scaled targets using the symbolic model
Y_val_pred_scaled = pysr_model.predict(X_train_np[300:])  # Change to pysr_model if using Option B

# Denormalize predictions back into physical units (nm)
Y_val_pred_physical = (Y_val_pred_scaled * np.array(Y_std)) + np.array(Y_mean)
Y_val_true_physical = (Y_train_np[300:] * np.array(Y_std)) + np.array(Y_mean)

# Calculate global validation error metrics
sr_rmse = np.sqrt(np.mean((Y_val_true_physical - Y_val_pred_physical) ** 2))
print(f"Global Validation Dataset RMSE: {sr_rmse:.4f} nm")

plt.figure(figsize=(12, 5))

# Plot 1: True vs Predicted Scatter Profile
plt.subplot(1, 2, 1)
target_mode = 0  # Focus on the designated mode (e.g., Defocus or Astigmatism)
plt.scatter(Y_val_true_physical[:, target_mode], Y_val_pred_physical[:, target_mode], alpha=0.4, color='teal')
plt.plot([Y_val_true_physical.min(), Y_val_true_physical.max()], 
         [Y_val_true_physical.min(), Y_val_true_physical.max()], 'r--', lw=2)
plt.xlabel('Simulated Truth Coefficients (nm)')
plt.ylabel('Symbolic Equation Reconstructed Output (nm)')
plt.title(f'Symbolic Regression Calibration: Mode #{target_mode}')
plt.grid(True, linestyle=':', alpha=0.6)

# Plot 2: Absolute Residual Error Map per Mode
plt.subplot(1, 2, 2)
residual_matrix = np.abs(Y_val_true_physical - Y_val_pred_physical)
mean_residuals = np.mean(residual_matrix, axis=0)
plt.bar(range(mean_residuals.shape[0]), mean_residuals, color='cadetblue', edgecolor='k', alpha=0.8)
plt.xlabel('Wavefront Mode Indices')
plt.ylabel('Mean Absolute Residual Error (nm)')
plt.title('Analytical Reconstruction Error Bounds')
plt.xticks(range(mean_residuals.shape[0]))
plt.grid(True, axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Extract the best SymPy expressions for each output mode
best_equations = pysr_model.get_best()

print("=== WINNING SYMBOLIC EQUATIONS PER MODE ===")
for mode_idx, eq in enumerate(best_equations):
    print(f"\nWavefront Mode {mode_idx}:")
    print(f"  Y_{mode_idx} = {eq}")

In [ ]:
from IPython.display import display, Math

print("=== BEAUTIFIED DISCOVERED FUNCTIONS ===")
# .latex() directly returns a list of raw LaTeX strings for each output mode
for mode_idx, latex_str in enumerate(pysr_model.latex()):
    display(Math(f"\\text{{Mode }}{mode_idx}: \\quad Y_{{{mode_idx}}} = {latex_str}"))